# 09 — V3-B Recall-First Tuning + Full Validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shelly-serafimovich/target-aware-infrared-sr/blob/main/notebooks/09_v3b_recall_tuning.ipynb)

Part A reproduces the 280-image tuning pilot.  
Part B freezes the selected configuration **V3-B = 3 guidance steps, LR = 0.02** and runs the full common validation set, then compares **Original DifIISR vs paired BASE vs selected V3-B** for image quality and frozen-YOLO detection.


In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=False)

from pathlib import Path
import os,sys,subprocess,json,re,hashlib,time
import numpy as np,pandas as pd,torch
from PIL import Image

PROJECT_REPO='https://github.com/shelly-serafimovich/target-aware-infrared-sr.git'
DIFIISR_REPO='https://github.com/zirui0625/DifIISR.git'
DIFIISR_COMMIT='09ca97ea48d481656dd8090e84099c963059ac41'
SEED=12345

DATASET_ROOT=Path('/content/drive/MyDrive/thermal-drone-dataset')
LR_DIR=DATASET_ROOT/'difiisr_baseline'/'valid'/'LR_difiisr_train_degradation_v2'
HR_DIR=DATASET_ROOT/'difiisr_baseline'/'valid'/'HR'
ORIG_SR=DATASET_ROOT/'difiisr_baseline'/'valid'/'SR_difiisr_train_degradation_v2'
ORIG_RESULTS=DATASET_ROOT/'difiisr_baseline'/'valid'/'results_difiisr_train_degradation_v2'
YOLO_WEIGHTS=DATASET_ROOT/'detector'/'weights'/'thermal_drone_yolo11n_corrected_v1_best.pt'

EXPERIMENT_ROOT=DATASET_ROOT/'target_aware_difiisr'/'v3_recall_pilot_v1'
TUNING_ROOT=DATASET_ROOT/'target_aware_difiisr'/'v3b_recall_tuning_v1'
FULL_ROOT=DATASET_ROOT/'target_aware_difiisr'/'v3b_full_validation_v1'
for p in [EXPERIMENT_ROOT,TUNING_ROOT,FULL_ROOT]: p.mkdir(parents=True,exist_ok=True)

project_dir=Path('/content/target-aware-infrared-sr')
difiisr_dir=Path('/content/DifIISR')

def run(cmd,cwd=None):
    print('$',' '.join(map(str,cmd)),flush=True)
    subprocess.run(list(map(str,cmd)),cwd=cwd,check=True)

if not project_dir.exists():
    run(['git','clone',PROJECT_REPO,project_dir])
else:
    run(['git','fetch','origin'],cwd=project_dir)
    run(['git','checkout','main'],cwd=project_dir)
    run(['git','pull','--ff-only','origin','main'],cwd=project_dir)

if not difiisr_dir.exists():
    run(['git','clone',DIFIISR_REPO,difiisr_dir])
run(['git','fetch','--all'],cwd=difiisr_dir)
run(['git','checkout','--detach',DIFIISR_COMMIT],cwd=difiisr_dir)

ANN_DIR=project_dir/'annotations'/'corrected_yolo'/'valid'/'labels'
ENV=Path('/content/difiisr-env'); PY=str(ENV/'bin'/'python')
if not Path(PY).exists():
    run([sys.executable,'-m','pip','install','-q','uv'])
    run([sys.executable,'-m','uv','venv','--python','3.10',ENV])
    run([sys.executable,'-m','uv','pip','install','--python',PY,
         'torch==2.1.1','torchvision==0.16.1','xformers==0.0.23','numpy==1.25.2',
         'scipy==1.9.3','packaging==24.2','setuptools==69.5.1'])
    run([sys.executable,'-m','uv','pip','install','--python',PY,
         'pyiqa==0.1.12','basicsr==1.4.2','opencv-python','matplotlib','timm','pandas',
         'pillow','scikit-learn','scikit-image','lpips','loguru','omegaconf','six',
         'tqdm','albumentations','einops','imageio','gdown'])
else:
    subprocess.run([PY,'-c','import pyiqa'],check=False)

run([sys.executable,'-m','pip','install','-q','ultralytics'])

weights=difiisr_dir/'weights'; weights.mkdir(exist_ok=True)
if not (weights/'DifIISR.pth').exists():
    run([PY,'-m','gdown','1PhRvk1Dlp3CCrPkrxRfNbNZ3fNgviDVZ','-O',weights/'DifIISR.pth'])
if not (weights/'autoencoder_vq_f4.pth').exists():
    run([PY,'-m','gdown','https://github.com/zsyOAOA/ResShift/releases/download/v2.0/autoencoder_vq_f4.pth','-O',weights/'autoencoder_vq_f4.pth'])

assert LR_DIR.exists() and HR_DIR.exists() and ORIG_SR.exists() and ANN_DIR.exists() and YOLO_WEIGHTS.exists()
print('✓ setup ready | GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')


## Part A — 280-image pilot tuning (already used to choose the final setting)

The selected pilot winner was **3 steps / LR 0.02**. These cells remain for reproducibility; you do not need to rerun the sweep before Part B if the pilot results already exist on Drive.


In [ ]:
PILOT_N_TARGET_IMAGES=200; PILOT_N_BACKGROUND_IMAGES=80
MANIFEST_PATH=EXPERIMENT_ROOT/f'pilot_manifest_{PILOT_N_TARGET_IMAGES}t_{PILOT_N_BACKGROUND_IMAGES}b_seed{SEED}.csv'
assert MANIFEST_PATH.exists(), f'Missing pilot manifest: {MANIFEST_PATH}'

COMMON=dict(seed=SEED,pilot_n_target_images=200,pilot_n_background_images=80,
    small_target_definition='upsampled pre-encoder bbox width<=24 AND height<=24',
    core=24,outer=48,top_k=15,sigma_floor=0.013458,blob_k=1.5,soft_t=0.25,soft_min_support=3.0,
    scr_gate=1.8,scr_gate_temp=0.25,nd_gate=0.40,nd_gate_temp=0.07,nd_floor_weight=0.50,
    max_candidates=48,proposal_nms=7,scr_goal=2.5,goal_temp=0.25,
    lambda_fid=10.0,lambda_bg=20.0,difiisr_commit=DIFIISR_COMMIT,
    primary_selection_metric='GT-matched recall @ IoU>=0.25 on <=24x24 targets')

SWEEP=[('s2_lr003',2,0.03),('s3_lr002',3,0.02),('s3_lr003',3,0.03),('s4_lr002',4,0.02)]
display(pd.DataFrame(SWEEP,columns=['name','steps','lr']))


In [ ]:
# Optional: reproduce/resume the tuning sweep.
RUNNER=project_dir/'scripts'/'v3_recall_guidance_runner.py'
DIFIISR_CONFIG=difiisr_dir/'configs'/'DifIISR_test.yaml'
runs=[]

for label,steps,lr in SWEEP:
    cfg=dict(COMMON,guidance_steps=steps,guidance_lr=lr)
    run_id='cfg_'+hashlib.sha1(json.dumps(cfg,sort_keys=True,separators=(',',':')).encode()).hexdigest()[:10]
    run_root=TUNING_ROOT/run_id; run_root.mkdir(parents=True,exist_ok=True)
    cfg_path=run_root/'config.json'; cfg_path.write_text(json.dumps(cfg,indent=2))
    cmd=[PY,str(RUNNER),'--manifest',str(MANIFEST_PATH),'--lr',str(LR_DIR),'--out',str(run_root),
         '--config',str(cfg_path),'--difiisr-config',str(DIFIISR_CONFIG)]
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout: print(line,end='',flush=True)
    if proc.wait(): raise RuntimeError(f'{label} generation failed')
    runs.append((label,steps,lr,run_id,run_root))

EVAL_SCRIPT=project_dir/'scripts'/'v3_recall_yolo_eval.py'
for label,steps,lr,run_id,run_root in runs:
    subprocess.run([sys.executable,str(EVAL_SCRIPT),'--manifest',str(MANIFEST_PATH),
                    '--run-root',str(run_root),'--weights',str(YOLO_WEIGHTS),
                    '--core','24','--iou-match','0.25','--fa-conf','0.25'],check=True)

rows=[]
for label,steps,lr,run_id,run_root in runs:
    er=run_root/'yolo_pilot_eval'
    sm=pd.read_csv(er/'pilot_summary.csv'); pr=pd.read_csv(er/'paired_saved_lost.csv')
    b=sm[sm.method=='V3-B'].iloc[0]; pp=pr[pr.method=='V3-B'].iloc[0]
    rows.append(dict(config=label,steps=steps,lr=lr,recall=b.small_target_recall_iou025,
        recall_delta_pp=b.recall_delta_pp_vs_BASE,mean_gt_conf=b.mean_gt_conf,
        saved=int(pp.saved),lost=int(pp.lost),net_saved=int(pp.net_saved),
        bg_images_with_FA=int(b.background_images_with_FA_conf025),
        bg_FA_count=int(b.background_FA_count_conf025),bg_max_conf_mean=b.background_max_conf_mean,
        run_id=run_id))
leader=pd.DataFrame(rows).sort_values(['recall','mean_gt_conf'],ascending=[False,False]).reset_index(drop=True)
display(leader)
leader.to_csv(TUNING_ROOT/'leaderboard.csv',index=False)


# Part B — FULL validation inference

We now **freeze the selected configuration**:
- V3-B only
- 3 guidance steps
- guidance LR = 0.02
- all other SCR / neighbour-density / fidelity settings unchanged from the pilot

The full comparison is:
1. **Original DifIISR** — the already-computed official frozen baseline.
2. **BASE** — paired unguided decode generated from exactly the same per-image `z0` / noise as our selected method.
3. **V3-B selected** — our bounded pooled-SCR + frozen neighbour-density guidance.

The known 480×270 LR outlier is excluded from the paired gradient-guidance run, exactly as in the previous full experiment, so every reported comparison uses the same common image set.


In [ ]:
FULL_MANIFEST=FULL_ROOT/'full_common_manifest.csv'
EXCLUDED=FULL_ROOT/'excluded_images.csv'

rows=[]; excluded=[]
for ip in sorted(p for p in LR_DIR.iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg'}):
    with Image.open(ip) as im:
        w,h=im.size
    lp=ANN_DIR/f'{ip.stem}.txt'
    is_bg=(not lp.exists()) or lp.stat().st_size==0
    rec=dict(image=ip.name,pilot_group='full_background' if is_bg else 'full_target',
             label_path=str(lp),is_background=is_bg,
             lr_width=w,lr_height=h,up_width=w*4,up_height=h*4)
    if (w,h)==(480,270) or (w,h)==(270,480):
        excluded.append(rec)
    else:
        rows.append(rec)

full_df=pd.DataFrame(rows)
pd.DataFrame(excluded).to_csv(EXCLUDED,index=False)
full_df.to_csv(FULL_MANIFEST,index=False)

print('Full common images:',len(full_df))
print('Target images:',int((~full_df.is_background).sum()))
print('Background images:',int(full_df.is_background.sum()))
print('Excluded for paired OOM:',len(excluded))
display(pd.DataFrame(excluded))


In [ ]:
FINAL_CONFIG=dict(COMMON,
    pilot_n_target_images=int((~full_df.is_background).sum()),
    pilot_n_background_images=int(full_df.is_background.sum()),
    guidance_steps=3,
    guidance_lr=0.02,
    selected_from_pilot='s3_lr002')

FINAL_RUN_ID='selected_s3_lr002'
FINAL_RUN_ROOT=FULL_ROOT/FINAL_RUN_ID
FINAL_RUN_ROOT.mkdir(parents=True,exist_ok=True)
FINAL_CONFIG_PATH=FINAL_RUN_ROOT/'config.json'
FINAL_CONFIG_PATH.write_text(json.dumps(FINAL_CONFIG,indent=2))

print('Selected full-run configuration:')
print('  steps =',FINAL_CONFIG['guidance_steps'])
print('  LR    =',FINAL_CONFIG['guidance_lr'])
print('  images=',len(full_df))
print('  output=',FINAL_RUN_ROOT)

cmd=[PY,str(RUNNER),'--manifest',str(FULL_MANIFEST),'--lr',str(LR_DIR),
     '--out',str(FINAL_RUN_ROOT),'--config',str(FINAL_CONFIG_PATH),
     '--difiisr-config',str(DIFIISR_CONFIG)]

proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout:
    print(line,end='',flush=True)
rc=proc.wait()
if rc:
    raise RuntimeError(f'Full V3-B inference stopped with code {rc}. Completed images are saved; rerun this cell to resume.')
print('✓ FULL paired inference complete')


## Part C — Image-quality comparison

This reproduces the quality family used in the earlier full pipeline:

**PSNR, SSIM, LPIPS, CLIP-IQA, MUSIQ, NIQE**

All methods are evaluated on the exact common paired image set. The official Original DifIISR metric table is reused when compatible; BASE and selected V3-B are evaluated with the same metric implementation.


In [ ]:
check=subprocess.run([PY,'-c','import pyiqa'],capture_output=True,text=True)
if check.returncode:
    run([sys.executable,'-m','uv','pip','install','--python',PY,'pyiqa==0.1.12'])

QUALITY_OUT=FULL_ROOT/'evaluation'/'quality'
QUALITY_OUT.mkdir(parents=True,exist_ok=True)
QUALITY_SCRIPT=project_dir/'scripts'/'v3_full_quality_eval.py'

baseline_candidates=[
    ORIG_RESULTS/'valid_difiisr_baseline.csv',
    ORIG_RESULTS/'valid_difiisr_per_image_metrics.csv'
]
ORIG_METRICS_CSV=next((p for p in baseline_candidates if p.exists()),Path(''))

cmd=[PY,str(QUALITY_SCRIPT),
     '--hr',str(HR_DIR),
     '--original',str(ORIG_SR),
     '--base',str(FINAL_RUN_ROOT/'BASE'),
     '--guided',str(FINAL_RUN_ROOT/'V3B_pooledSCR_ND'),
     '--out',str(QUALITY_OUT)]
if str(ORIG_METRICS_CSV):
    cmd += ['--original-csv',str(ORIG_METRICS_CSV)]

proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout: print(line,end='',flush=True)
if proc.wait():
    raise RuntimeError('Quality evaluation stopped. CSV checkpoints are saved; rerun to resume.')

print('✓ quality metrics complete')


## Part D — Frozen-YOLO comparison

The exact same frozen detector is run on all three image sets.

Outputs include:
- overall GT-matched recall @ IoU ≥ 0.25
- recall on the ≤24×24 small-target regime
- mean GT confidence
- paired **saved / lost** targets vs BASE
- background images with false alarms @ conf ≥ 0.25
- total background false-alarm count
- standard Ultralytics **precision, recall, AP50, mAP50-95**


In [ ]:
DET_OUT=FULL_ROOT/'evaluation'/'detector'
DET_OUT.mkdir(parents=True,exist_ok=True)
FULL_YOLO_SCRIPT=project_dir/'scripts'/'v3_full_yolo_eval.py'

cmd=[sys.executable,str(FULL_YOLO_SCRIPT),
     '--manifest',str(FULL_MANIFEST),
     '--original',str(ORIG_SR),
     '--base',str(FINAL_RUN_ROOT/'BASE'),
     '--guided',str(FINAL_RUN_ROOT/'V3B_pooledSCR_ND'),
     '--weights',str(YOLO_WEIGHTS),
     '--out',str(DET_OUT),
     '--iou-match','0.25',
     '--fa-conf','0.25']

proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout: print(line,end='',flush=True)
if proc.wait():
    raise RuntimeError('Detector evaluation stopped. Per-image caches are saved; rerun to resume.')

print('✓ detector evaluation complete')


## Part E — Final comparison table


In [ ]:
quality=pd.read_csv(QUALITY_OUT/'quality_all_methods.csv')
qsum=quality.groupby('method')[['psnr','ssim','lpips','clipiqa','musiq','niqe']].mean().reset_index()

det=pd.read_csv(DET_OUT/'detection_summary.csv')
paired=pd.read_csv(DET_OUT/'paired_saved_lost.csv')

final=det.merge(qsum,on='method',how='left')
preferred=[
    'method','n_images','n_targets','target_recall_iou025',
    'n_small_targets','small_target_recall_iou025','mean_gt_conf_small',
    'precision','recall','AP50','mAP50_95',
    'n_background_images','background_images_with_FA_conf025',
    'background_FA_count_conf025','background_max_conf_mean',
    'psnr','ssim','lpips','clipiqa','musiq','niqe'
]
final=final[[c for c in preferred if c in final.columns]]
display(final)

print('\nPAIRED SAVED / LOST VS BASE')
display(paired)

final.to_csv(FULL_ROOT/'FINAL_COMPARISON.csv',index=False)
print('Saved final table:',FULL_ROOT/'FINAL_COMPARISON.csv')
